In [ ]:
!pip install -q datasets transformers sentencepiece sacrebleu scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import html

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

In [ ]:
dataset = load_dataset(
    "akbargherbal/ONE_MILLION_AR_TO_EN_SENTENCES_DATASET",
    split="train[:40000]"
)

print(dataset)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 40000
})


In [ ]:
print("Dataset columns:")
print(dataset.column_names)

print("\nFirst example:")
print(dataset[0])

print("\nNumber of examples:")
print(len(dataset))

Dataset columns:
['input', 'output', 'instruction']

First example:
{'input': 'وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص كل من الدستور والقوانين ذات الصلة صراحة على الظروف التي تُفرض فيها قيود.', 'output': 'Previous censorship was outlawed, and both the Constitution and relevant laws expressly state under which circumstances restrictions may be imposed.', 'instruction': 'Convert the following Arabic text into English.'}

Number of examples:
40000


In [ ]:
df = dataset.to_pandas()

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (40000, 3)


,input,output,instruction
0,وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص ك...,"Previous censorship was outlawed, and both the...",Convert the following Arabic text into English.
1,وبحسب البرامج الزمنية المنصوص عليها في العقد، ...,"According to the contract &apos; s schedules, ...",Please translate the given Arabic sentence int...
2,- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجز...,persons who reached pensionable age or were re...,Change the following Arabic phrase to English.
3,190 - ومسؤولية تزويد العاملين في الميدان بما ي...,190. The responsibility for providing the peop...,Turn the Arabic sentence below into English.
4,٢٠ - السيد مكتفي )الجزائر(: أعرب عن تأييده للب...,20. Mr. Moktefi (Algeria) supported the statem...,Provide an English translation for the followi...


In [ ]:
print("Columns:")
print(df.columns.tolist())

Columns:
['input', 'output', 'instruction']


In [ ]:
df = df[["input", "output"]].copy()

df = df.rename(
    columns={
        "output": "english_text",
        "input": "arabic_text"
    }
)

df.head()

,arabic_text,english_text
0,وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص ك...,"Previous censorship was outlawed, and both the..."
1,وبحسب البرامج الزمنية المنصوص عليها في العقد، ...,"According to the contract &apos; s schedules, ..."
2,- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجز...,persons who reached pensionable age or were re...
3,190 - ومسؤولية تزويد العاملين في الميدان بما ي...,190. The responsibility for providing the peop...
4,٢٠ - السيد مكتفي )الجزائر(: أعرب عن تأييده للب...,20. Mr. Moktefi (Algeria) supported the statem...


In [ ]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
arabic_text     0
english_text    0
dtype: int64


In [ ]:
df["english_text"] = df["english_text"].astype(str).apply(html.unescape)
df["arabic_text"] = df["arabic_text"].astype(str).apply(html.unescape)

print("HTML entities cleaned.")

HTML entities cleaned.


In [ ]:
df["english_text"] = df["english_text"].str.strip()
df["arabic_text"] = df["arabic_text"].str.strip()

In [ ]:
df = df.dropna(
    subset=["english_text", "arabic_text"]
).copy()

df = df[
    (df["english_text"] != "") &
    (df["arabic_text"] != "")
].copy()

df = df.reset_index(drop=True)

print("Dataset size after cleaning:", len(df))

Dataset size after cleaning: 40000


In [ ]:
print(
    "Duplicate English-Arabic pairs:",
    df.duplicated(
        subset=["english_text", "arabic_text"]
    ).sum()
)

Duplicate English-Arabic pairs: 0


In [ ]:
# Remove duplicate English-Arabic pairs
df = df.drop_duplicates(
    subset=["english_text", "arabic_text"]
).reset_index(drop=True)

print("Dataset size after duplicate removal:", len(df))

Dataset size after duplicate removal: 40000


In [ ]:
print("Final cleaning checks:")

print("Missing English:", df["english_text"].isnull().sum())
print("Missing Arabic:", df["arabic_text"].isnull().sum())

print("Empty English:", (df["english_text"] == "").sum())
print("Empty Arabic:", (df["arabic_text"] == "").sum())

print(
    "Duplicate pairs:",
    df.duplicated(
        subset=["english_text", "arabic_text"]
    ).sum()
)

Final cleaning checks:
Missing English: 0
Missing Arabic: 0
Empty English: 0
Empty Arabic: 0
Duplicate pairs: 0


In [ ]:
for i in range(3):
    print("=" * 80)

    print("English source:")
    print(df.loc[i, "english_text"])

    print("\nArabic target:")
    print(df.loc[i, "arabic_text"])

    print()

English source:
Previous censorship was outlawed, and both the Constitution and relevant laws expressly state under which circumstances restrictions may be imposed.

Arabic target:
وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص كل من الدستور والقوانين ذات الصلة صراحة على الظروف التي تُفرض فيها قيود.

English source:
According to the contract ' s schedules, approximately 92 per cent of the contract value related to the supply of goods, and approximately 8 per cent related to services.

Arabic target:
وبحسب البرامج الزمنية المنصوص عليها في العقد، فإن نسبة نحو 92 في المائة من قيمة العقد تتصل بإمداد المواد وما يقرب من 8 في المائة بالخدمات.

English source:
persons who reached pensionable age or were recognized as disabled while raising children of the deceased person who were receiving or were entitled to receive orphan ' s (survivor ' s) pension.

Arabic target:
- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجزين أثناء قيامهم بتربية أطفال الشخص المتوفى الذين يحصلون أو يحق لهم أن يتقاضوا معاش ا

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

print(
    "\nTotal examples:",
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

Training examples: 32000
Validation examples: 4000
Test examples: 4000

Total examples: 40000


In [ ]:
# Verify that the data split is correct
assert len(train_df) + len(validation_df) + len(test_df) == len(df)

# Verify that no missing values exist
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    assert split_df["english_text"].isnull().sum() == 0
    assert split_df["arabic_text"].isnull().sum() == 0
    assert (split_df["english_text"] == "").sum() == 0
    assert (split_df["arabic_text"] == "").sum() == 0

print("All data split checks passed successfully.")

All data split checks passed successfully.


In [ ]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

prepared_dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})

print(prepared_dataset)

DatasetDict({
    train: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 32000
    })
    validation: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 4000
    })
})


In [ ]:
print("Final columns:")
print(prepared_dataset["train"].column_names)

print("\nExample prepared training sample:")
print(prepared_dataset["train"][0])

Final columns:
['arabic_text', 'english_text']

Example prepared training sample:
{'arabic_text': 'وتحقيقاً لذلك، قام الممثل بزيارة مخيمات ومستوطنات المشردين داخلياً في الخرطوم وحولها في شيكان، والفتح 2 ومايو، واتجه إلى أبيي وكادوغلي وملكال وملوالكون ورومبك وجوبا.', 'english_text': 'To this end, the Representative visited Khartoum and the surrounding IDP camps and settlements at Shikan, Al Fatah 3 and Mayo, and travelled to Abyei, Kadugli, Malakal, Malualkon, Rumbek and Juba.'}


In [ ]:
print("Translation direction verification:")

print("\nEnglish source:")
print(prepared_dataset["train"][0]["english_text"])

print("\nArabic target:")
print(prepared_dataset["train"][0]["arabic_text"])

Translation direction verification:

English source:
To this end, the Representative visited Khartoum and the surrounding IDP camps and settlements at Shikan, Al Fatah 3 and Mayo, and travelled to Abyei, Kadugli, Malakal, Malualkon, Rumbek and Juba.

Arabic target:
وتحقيقاً لذلك، قام الممثل بزيارة مخيمات ومستوطنات المشردين داخلياً في الخرطوم وحولها في شيكان، والفتح 2 ومايو، واتجه إلى أبيي وكادوغلي وملكال وملوالكون ورومبك وجوبا.


In [ ]:
print("=" * 60)
print("PART 1: SETUP + DATA PREPARATION COMPLETED")
print("=" * 60)

print("\nDataset splits:")
print(
    f"Train: {len(train_df)} "
    f"({len(train_df) / len(df) * 100:.1f}%)"
)
print(
    f"Validation: {len(validation_df)} "
    f"({len(validation_df) / len(df) * 100:.1f}%)"
)
print(
    f"Test: {len(test_df)} "
    f"({len(test_df) / len(df) * 100:.1f}%)"
)

print("\nTotal samples:")
print(len(df))

print("\nFinal columns:")
print(prepared_dataset["train"].column_names)

print("\nTranslation direction:")
print("English source -> Arabic target")

print("\nPart 1 completed successfully.")

PART 1: SETUP + DATA PREPARATION COMPLETED

Dataset splits:
Train: 32000 (80.0%)
Validation: 4000 (10.0%)
Test: 4000 (10.0%)

Total samples:
40000

Final columns:
['arabic_text', 'english_text']

Translation direction:
English source -> Arabic target

Part 1 completed successfully.
